In [1]:
!pip install xgboost

In [2]:
# ========= 1) Imports y configuración =========


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42

from google.colab import drive
drive.mount('/content/drive')



#Metricas de evaluación
from sklearn.metrics import (
    make_scorer,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

#Modelos de clasificación
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier



#Librerias para optimizar parametros, balancear y pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GroupShuffleSplit
from collections import Counter
from imblearn.pipeline import Pipeline as ImbPipeline
import time

Mounted at /content/drive


In [3]:
# ========= 2) Cargar CSV =========
import pandas as pd

ruta = '/content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Datasets/df_procesado_bucaramanga.csv'
df = pd.read_csv(ruta)

print("Shape:", df.shape)
print("Columnas:", df.columns.tolist())


Shape: (108538, 31)
Columnas: ['artritis', 'diabetes', 'hipertensi_n', 'epoc', 'asma', 'insuficiencia_cardiaca', 'c_ncer', 'huerfanas_hemofilias_y_otras', 'cirugia_cardiaca', 'trasplantados', 'compromiso_renal', 'sexo_MASCULINO', 'ciclo_de_vida_ADULTEZ', 'ciclo_de_vida_INFANCIA', 'ciclo_de_vida_JOVENES', 'ciclo_de_vida_PERSONA MAYOR', 'ciclo_de_vida_PRIMERA INFANCIA', 'tipo_erc_limpio_Estadio II', 'tipo_erc_limpio_Estadio III', 'tipo_erc_limpio_Estadio IV', 'tipo_erc_limpio_Estadio V', 'tipo_erc_limpio_SIN CLAS', 'erc_trr_limpio_Hemodiálisis', 'erc_trr_limpio_No recibe TRR', 'erc_trr_limpio_Prediálisis', 'erc_trr_limpio_Recibe TRR (no especificada)', 'erc_trr_limpio_Registro inconsistente', 'erc_trr_limpio_Trasplante (no definido / no disponible)', 'regimen_E', 'regimen_P', 'regimen_S']


In [4]:
# ========= 3) Separar features y target =========
target = "diabetes"
feature_cols = [c for c in df.columns if c != target]

X = df[feature_cols].copy()
y = df[target].copy()

print("Distribución de la variable objetivo:")
print(y.value_counts(normalize=True).rename("proporción"))


Distribución de la variable objetivo:
diabetes
0    0.645709
1    0.354291
Name: proporción, dtype: float64


In [5]:
# ========= 4) Split SIN FUGA (agrupa por perfil) =========
groups = pd.util.hash_pandas_object(X, index=False)

gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print("\n--- División Realizada (Antes de SMOTE) ---")
print(f"Train (Original): {X_train.shape} | Clases: {Counter(y_train)}")
print(f"Test (Original):  {X_test.shape}  | Clases: {Counter(y_test)}")




--- División Realizada (Antes de SMOTE) ---
Train (Original): (73613, 30) | Clases: Counter({0: 46655, 1: 26958})
Test (Original):  (34925, 30)  | Clases: Counter({0: 23429, 1: 11496})


# Regresión Logistica

In [ ]:


# ============================================
# 1) OPTIMIZACIÓN DE HIPERPARÁMETROS (GRIDSEARCH CON PIPELINE)
# ============================================

# 1. Definimos el Pipeline
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

# 2. Grilla de hiperparámetros
param_grid_lr = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['lbfgs', 'liblinear'],
    'model__penalty': ['l2']
}

# 3. Configuración de la búsqueda
grid_lr = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid_lr,
    scoring=make_scorer(f1_score),
    cv=5,
    n_jobs=-1,
    verbose=1
)

# 4. Entrenar búsqueda (MEDICIÓN DE TIEMPO)
print("⏳ Iniciando entrenamiento...")
start_train = time.time()  # <--- ⏱️ Marca de inicio

grid_lr.fit(X_train, y_train)

end_train = time.time()    # <--- ⏱️ Marca de fin
train_time = end_train - start_train # Cálculo

# Seleccionar el mejor modelo
best_lr = grid_lr.best_estimator_

print(f"✅ Entrenamiento finalizado en {train_time:.2f} segundos.") # Imprimir tiempo
print("🔹 Mejores hiperparámetros:", grid_lr.best_params_)
print("🔹 Mejor F1 promedio (CV Realista):", grid_lr.best_score_)


# ============================================
# 2) EVALUACIÓN EN TEST
# ============================================

# Calcular probabilidades (MEDICIÓN DE TIEMPO DE PREDICCIÓN)
# Esto es útil para saber qué tan rápido es el modelo respondiendo
start_pred = time.time() # <--- ⏱️ Inicio predicción

y_proba_lr = best_lr.predict_proba(X_test)[:, 1]

end_pred = time.time()   # <--- ⏱️ Fin predicción
pred_time = end_pred - start_pred

print(f"⚡ Tiempo de predicción (Inferencia) en Test: {pred_time:.4f} segundos.")

# Definir los umbrales
umbrales = [0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
resultados_lr = []

for thr in umbrales:
    y_pred_lr = (y_proba_lr >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_lr).ravel()

    resultados_lr.append({
        "Umbral": thr,
        "Accuracy": accuracy_score(y_test, y_pred_lr),
        "Precision": precision_score(y_test, y_pred_lr),
        "Recall": recall_score(y_test, y_pred_lr),
        "F1": f1_score(y_test, y_pred_lr),
        "ROC-AUC": roc_auc_score(y_test, y_proba_lr),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Training Time (s)": round(train_time, 2), # <--- Agregado a la tabla
        "Inference Time (s)": round(pred_time, 4),  # <--- Agregado a la tabla
        "Mejor F1 promedio (CV Realista)": grid_lr.best_score_,
    })

# Crear tabla final
tabla_resultados_lr = pd.DataFrame(resultados_lr).round(4)
tabla_resultados_lr

⏳ Iniciando entrenamiento...
Fitting 5 folds for each of 8 candidates, totalling 40 fits
✅ Entrenamiento finalizado en 100.01 segundos.
🔹 Mejores hiperparámetros: {'model__C': 1, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
🔹 Mejor F1 promedio (CV Realista): 0.55695933576072
⚡ Tiempo de predicción (Inferencia) en Test: 0.0080 segundos.


,Umbral,Accuracy,Precision,Recall,F1,ROC-AUC,TP,FP,FN,TN,Training Time (s),Inference Time (s),Mejor F1 promedio (CV Realista)
0,0.7,0.6835,0.6434,0.0865,0.1524,0.6408,994,551,10502,22878,100.01,0.008,0.557
1,0.6,0.7308,0.7034,0.3152,0.4353,0.6408,3623,1528,7873,21901,100.01,0.008,0.557
2,0.5,0.5301,0.3785,0.6659,0.4826,0.6408,7655,12571,3841,10858,100.01,0.008,0.557
3,0.4,0.3966,0.3453,0.9300,0.5036,0.6408,10691,20269,805,3160,100.01,0.008,0.557
4,0.3,0.3682,0.3397,0.9741,0.5037,0.6408,11198,21768,298,1661,100.01,0.008,0.557
5,0.2,0.3606,0.3379,0.9825,0.5029,0.6408,11295,22131,201,1298,100.01,0.008,0.557
6,0.1,0.3435,0.3330,0.9911,0.4985,0.6408,11394,22827,102,602,100.01,0.008,0.557


# Árbol de decision

In [ ]:

# ============================================
# 1) OPTIMIZACIÓN DE HIPERPARÁMETROS (DECISION TREE CON PIPELINE)
# ============================================

# 1. Definimos el Pipeline
pipeline_dt = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

# 2. Grilla de hiperparámetros
param_grid_dt = {
    'model__max_depth': [3, 4, 5, 6, 8, 10],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__criterion': ['gini', 'entropy']
}

# 3. Configuración de la búsqueda
grid_dt = GridSearchCV(
    estimator=pipeline_dt,
    param_grid=param_grid_dt,
    scoring=make_scorer(f1_score),
    cv=5,
    n_jobs=-1,
    verbose=1
)

# 4. Entrenar búsqueda (CON MEDICIÓN DE TIEMPO)
print("⏳ Iniciando entrenamiento Decision Tree...")
start_train = time.time()  # <--- ⏱️ Inicio

grid_dt.fit(X_train, y_train)

end_train = time.time()    # <--- ⏱️ Fin
train_time = end_train - start_train

# Guardar el mejor modelo
best_dt = grid_dt.best_estimator_

print(f"✅ Entrenamiento finalizado en {train_time:.2f} segundos.")
print("🔹 Mejores hiperparámetros:", grid_dt.best_params_)
print("🔹 Mejor F1 promedio (CV Realista):", grid_dt.best_score_)


# ============================================
# 2) EVALUACIÓN EN TEST CON UMBRALES VARIABLES
# ============================================

# Calcular probabilidades (CON MEDICIÓN DE TIEMPO)
start_pred = time.time()   # <--- ⏱️ Inicio Inferencia

y_proba_dt = best_dt.predict_proba(X_test)[:, 1]

end_pred = time.time()     # <--- ⏱️ Fin Inferencia
pred_time = end_pred - start_pred

print(f"⚡ Tiempo de predicción (Inferencia) en Test: {pred_time:.4f} segundos.")

# Definir los umbrales a evaluar
umbrales = [0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]

resultados_dt = []

for thr in umbrales:
    y_pred_dt = (y_proba_dt >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_dt).ravel()

    resultados_dt.append({
        "Umbral": thr,
        "Accuracy": accuracy_score(y_test, y_pred_dt),
        "Precision": precision_score(y_test, y_pred_dt),
        "Recall": recall_score(y_test, y_pred_dt),
        "F1": f1_score(y_test, y_pred_dt),
        "ROC-AUC": roc_auc_score(y_test, y_proba_dt),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Training Time (s)": round(train_time, 2), # <--- Agregado
        "Inference Time (s)": round(pred_time, 4),  # <--- Agregado
        "Mejor F1 promedio (CV Realista)": grid_dt.best_score_,
    })

# Crear tabla final
tabla_resultados_dt = pd.DataFrame(resultados_dt).round(4)
tabla_resultados_dt


⏳ Iniciando entrenamiento Decision Tree...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
✅ Entrenamiento finalizado en 1093.23 segundos.
🔹 Mejores hiperparámetros: {'model__criterion': 'entropy', 'model__max_depth': 8, 'model__min_samples_leaf': 4, 'model__min_samples_split': 10}
🔹 Mejor F1 promedio (CV Realista): 0.5555601941631549
⚡ Tiempo de predicción (Inferencia) en Test: 0.0058 segundos.


,Umbral,Accuracy,Precision,Recall,F1,ROC-AUC,TP,FP,FN,TN,Training Time (s),Inference Time (s),Mejor F1 promedio (CV Realista)
0,0.7,0.7344,0.7134,0.3230,0.4446,0.6689,3713,1492,7783,21937,1093.23,0.0058,0.5556
1,0.6,0.7354,0.7015,0.3416,0.4595,0.6689,3927,1671,7569,21758,1093.23,0.0058,0.5556
2,0.5,0.7351,0.6900,0.3543,0.4682,0.6689,4073,1830,7423,21599,1093.23,0.0058,0.5556
3,0.4,0.4483,0.3698,0.9596,0.5338,0.6689,11031,18802,465,4627,1093.23,0.0058,0.5556
4,0.3,0.4474,0.3697,0.9632,0.5343,0.6689,11073,18876,423,4553,1093.23,0.0058,0.5556
5,0.2,0.4269,0.3621,0.9727,0.5277,0.6689,11182,19702,314,3727,1093.23,0.0058,0.5556
6,0.1,0.4129,0.3575,0.9835,0.5244,0.6689,11306,20316,190,3113,1093.23,0.0058,0.5556


# Arboles Aleatorios

In [ ]:

# ============================================
# 1) OPTIMIZACIÓN DE HIPERPARÁMETROS (RANDOM FOREST CON PIPELINE)
# ============================================

# 1. Definimos el Pipeline
# Primero SMOTE (solo train fold), luego RandomForest
pipeline_rf = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

# 2. Grilla de hiperparámetros
# Agregamos el prefijo 'model__' porque el RF está dentro del paso 'model'
param_grid_rf = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [4, 6, 8, 10],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__criterion': ['gini', 'entropy']
}

# 3. Configuración de la búsqueda
grid_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    scoring=make_scorer(f1_score),
    cv=5,
    n_jobs=-1,
    verbose=1
)

# 4. Entrenar búsqueda (CON MEDICIÓN DE TIEMPO)
print("⏳ Iniciando entrenamiento Random Forest...")
start_train = time.time()  # <--- ⏱️ Inicio Entrenamiento

grid_rf.fit(X_train, y_train)

end_train = time.time()    # <--- ⏱️ Fin Entrenamiento
train_time = end_train - start_train

# Guardar el mejor modelo (Pipeline ajustado)
best_rf = grid_rf.best_estimator_

print(f"✅ Entrenamiento finalizado en {train_time:.2f} segundos.")
print("🔹 Mejores hiperparámetros:", grid_rf.best_params_)
print("🔹 Mejor F1 promedio (CV Realista):", grid_rf.best_score_)


# ============================================
# 2) EVALUACIÓN EN TEST CON UMBRALES VARIABLES
# ============================================

# Calcular probabilidades en el TEST REAL (X_test original)
print("⏳ Calculando probabilidades en Test...")
start_pred = time.time()   # <--- ⏱️ Inicio Inferencia

y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

end_pred = time.time()     # <--- ⏱️ Fin Inferencia
pred_time = end_pred - start_pred

print(f"⚡ Tiempo de predicción (Inferencia) en Test: {pred_time:.4f} segundos.")

# Definir los umbrales
umbrales = [0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]

resultados_rf = []

for thr in umbrales:
    y_pred_rf = (y_proba_rf >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_rf).ravel()

    resultados_rf.append({
        "Umbral": thr,
        "Accuracy": accuracy_score(y_test, y_pred_rf),
        "Precision": precision_score(y_test, y_pred_rf),
        "Recall": recall_score(y_test, y_pred_rf),
        "F1": f1_score(y_test, y_pred_rf),
        "ROC-AUC": roc_auc_score(y_test, y_proba_rf),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Training Time (s)": round(train_time, 2), # <--- Agregado
        "Inference Time (s)": round(pred_time, 4),  # <--- Agregado
        "Mejor F1 promedio (CV Realista)": grid_rf.best_score_,
    })

# Crear tabla final
tabla_resultados_rf = pd.DataFrame(resultados_rf).round(4)
tabla_resultados_rf

⏳ Iniciando entrenamiento Random Forest...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
✅ Entrenamiento finalizado en 5075.97 segundos.
🔹 Mejores hiperparámetros: {'model__criterion': 'entropy', 'model__max_depth': 8, 'model__min_samples_leaf': 2, 'model__min_samples_split': 10, 'model__n_estimators': 100}
🔹 Mejor F1 promedio (CV Realista): 0.5741412837542545
⏳ Calculando probabilidades en Test...
⚡ Tiempo de predicción (Inferencia) en Test: 0.0849 segundos.


,Umbral,Accuracy,Precision,Recall,F1,ROC-AUC,TP,FP,FN,TN,Training Time (s),Inference Time (s),Mejor F1 promedio (CV Realista)
0,0.7,0.6747,0.6611,0.0243,0.0468,0.6464,279,143,11217,23286,5075.97,0.0849,0.5741
1,0.6,0.7347,0.7502,0.2911,0.4194,0.6464,3346,1114,8150,22315,5075.97,0.0849,0.5741
2,0.5,0.4559,0.3701,0.9302,0.5295,0.6464,10694,18202,802,5227,5075.97,0.0849,0.5741
3,0.4,0.4175,0.3587,0.9773,0.5248,0.6464,11235,20084,261,3345,5075.97,0.0849,0.5741
4,0.3,0.3534,0.3371,0.9980,0.5040,0.6464,11473,22561,23,868,5075.97,0.0849,0.5741
5,0.2,0.3328,0.3304,1.0000,0.4967,0.6464,11496,23302,0,127,5075.97,0.0849,0.5741
6,0.1,0.3292,0.3292,1.0000,0.4953,0.6464,11496,23427,0,2,5075.97,0.0849,0.5741


# Gradient Bossting

In [ ]:
# ============================================
# 1) OPTIMIZACIÓN DE HIPERPARÁMETROS (GRADIENT BOOSTING CON PIPELINE)
# ============================================

# 1. Definimos el Pipeline Seguro
pipeline_gb = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))
])

# 2. Grilla de hiperparámetros
param_grid_gb = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__max_depth': [3, 4, 5],
    'model__subsample': [0.8, 1.0]
}

# 3. Configuración de la búsqueda
grid_gb = GridSearchCV(
    estimator=pipeline_gb,
    param_grid=param_grid_gb,
    scoring=make_scorer(f1_score), # Prioriza F1
    cv=5,
    n_jobs=-1,
    verbose=1
)

# 4. Entrenar búsqueda (CON MEDICIÓN DE TIEMPO)
print("⏳ Iniciando entrenamiento Gradient Boosting...")
start_train = time.time()  # <--- ⏱️ Inicio Entrenamiento

grid_gb.fit(X_train, y_train)

end_train = time.time()    # <--- ⏱️ Fin Entrenamiento
train_time = end_train - start_train

# Guardar el mejor modelo
best_gb = grid_gb.best_estimator_

print(f"✅ Entrenamiento finalizado en {train_time:.2f} segundos.")
print("🔹 Mejores hiperparámetros:", grid_gb.best_params_)
print("🔹 Mejor F1 promedio (CV Realista):", grid_gb.best_score_)


# ============================================
# 2) EVALUACIÓN EN TEST CON UMBRALES VARIABLES
# ============================================

# Calcular probabilidades (CON MEDICIÓN DE TIEMPO)
print("⏳ Calculando probabilidades en Test...")
start_pred = time.time()   # <--- ⏱️ Inicio Inferencia

y_proba_gb = best_gb.predict_proba(X_test)[:, 1]

end_pred = time.time()     # <--- ⏱️ Fin Inferencia
pred_time = end_pred - start_pred

print(f"⚡ Tiempo de predicción (Inferencia) en Test: {pred_time:.4f} segundos.")

# Definir los umbrales
umbrales = [0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]

resultados_gb = []

for thr in umbrales:
    y_pred_gb = (y_proba_gb >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_gb).ravel()

    resultados_gb.append({
        "Umbral": thr,
        "Accuracy": accuracy_score(y_test, y_pred_gb),
        "Precision": precision_score(y_test, y_pred_gb),
        "Recall": recall_score(y_test, y_pred_gb),
        "F1": f1_score(y_test, y_pred_gb),
        "ROC-AUC": roc_auc_score(y_test, y_proba_gb),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Training Time (s)": round(train_time, 2), # <--- Agregado
        "Inference Time (s)": round(pred_time, 4),  # <--- Agregado
        "Mejor F1 promedio (CV Realista)": grid_gb.best_score_,
    })

# Crear tabla final
tabla_resultados_gb = pd.DataFrame(resultados_gb).round(4)
tabla_resultados_gb

⏳ Iniciando entrenamiento Gradient Boosting...
Fitting 5 folds for each of 54 candidates, totalling 270 fits
✅ Entrenamiento finalizado en 2752.09 segundos.
🔹 Mejores hiperparámetros: {'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__n_estimators': 100, 'model__subsample': 0.8}
🔹 Mejor F1 promedio (CV Realista): 0.5696663139528311
⏳ Calculando probabilidades en Test...
⚡ Tiempo de predicción (Inferencia) en Test: 0.0399 segundos.


,Umbral,Accuracy,Precision,Recall,F1,ROC-AUC,TP,FP,FN,TN,Training Time (s),Inference Time (s),Mejor F1 promedio (CV Realista)
0,0.7,0.7390,0.7518,0.3091,0.4380,0.6711,3553,1173,7943,22256,2752.09,0.0399,0.5697
1,0.6,0.7394,0.7292,0.3316,0.4559,0.6711,3812,1416,7684,22013,2752.09,0.0399,0.5697
2,0.5,0.4665,0.3732,0.9140,0.5300,0.6711,10507,17645,989,5784,2752.09,0.0399,0.5697
3,0.4,0.4482,0.3695,0.9576,0.5332,0.6711,11008,18783,488,4646,2752.09,0.0399,0.5697
4,0.3,0.4323,0.3646,0.9751,0.5307,0.6711,11210,19540,286,3889,2752.09,0.0399,0.5697
5,0.2,0.3828,0.3472,0.9937,0.5146,0.6711,11424,21482,72,1947,2752.09,0.0399,0.5697
6,0.1,0.3405,0.3329,0.9999,0.4995,0.6711,11495,23031,1,398,2752.09,0.0399,0.5697


# XGboost

In [ ]:


# ============================================
# 1) OPTIMIZACIÓN DE HIPERPARÁMETROS (XGBOOST CON PIPELINE)
# ============================================

# 1. Definimos el Pipeline Seguro
pipeline_xgb = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        use_label_encoder=False,
        eval_metric='logloss'
    ))
])

# 2. Grilla de hiperparámetros
# IMPORTANTE: Prefijo 'model__' para todos
param_grid_xgb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 4, 5],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

# 3. Configuración del grid search
grid_xgb = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid_xgb,
    scoring=make_scorer(f1_score), # Prioriza F1
    cv=5,
    n_jobs=-1,
    verbose=1
)

# 4. Entrenar búsqueda (CON MEDICIÓN DE TIEMPO)
print("⏳ Iniciando entrenamiento XGBoost...")
start_train = time.time()  # <--- ⏱️ Inicio Entrenamiento

grid_xgb.fit(X_train, y_train)

end_train = time.time()    # <--- ⏱️ Fin Entrenamiento
train_time = end_train - start_train

# Guardar el mejor modelo
best_xgb = grid_xgb.best_estimator_

print(f"✅ Entrenamiento finalizado en {train_time:.2f} segundos.")
print("🔹 Mejores hiperparámetros:", grid_xgb.best_params_)
print("🔹 Mejor F1 promedio (CV Realista):", grid_xgb.best_score_)


# ============================================
# 2) EVALUACIÓN EN TEST CON UMBRALES VARIABLES
# ============================================

# Calcular probabilidades (CON MEDICIÓN DE TIEMPO)
print("⏳ Calculando probabilidades en Test...")
start_pred = time.time()   # <--- ⏱️ Inicio Inferencia

y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]

end_pred = time.time()     # <--- ⏱️ Fin Inferencia
pred_time = end_pred - start_pred

print(f"⚡ Tiempo de predicción (Inferencia) en Test: {pred_time:.4f} segundos.")

# Definir umbrales
umbrales = [0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
resultados_xgb = []

for thr in umbrales:
    y_pred_xgb = (y_proba_xgb >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_xgb).ravel()

    resultados_xgb.append({
        "Umbral": thr,
        "Accuracy": accuracy_score(y_test, y_pred_xgb),
        "Precision": precision_score(y_test, y_pred_xgb),
        "Recall": recall_score(y_test, y_pred_xgb),
        "F1": f1_score(y_test, y_pred_xgb),
        "ROC-AUC": roc_auc_score(y_test, y_proba_xgb),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Training Time (s)": round(train_time, 2), # <--- Agregado
        "Inference Time (s)": round(pred_time, 4),  # <--- Agregado
        "Mejor F1 promedio (CV Realista)": grid_xgb.best_score_,
    })

# Crear tabla final
tabla_resultados_xgb = pd.DataFrame(resultados_xgb).round(4)
tabla_resultados_xgb

⏳ Iniciando entrenamiento XGBoost...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
✅ Entrenamiento finalizado en 1504.14 segundos.
🔹 Mejores hiperparámetros: {'model__colsample_bytree': 1.0, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 200, 'model__subsample': 1.0}
🔹 Mejor F1 promedio (CV Realista): 0.574152051221345
⏳ Calculando probabilidades en Test...
⚡ Tiempo de predicción (Inferencia) en Test: 0.0594 segundos.


,Umbral,Accuracy,Precision,Recall,F1,ROC-AUC,TP,FP,FN,TN,Training Time (s),Inference Time (s),Mejor F1 promedio (CV Realista)
0,0.7,0.7330,0.7772,0.2649,0.3951,0.6759,3045,873,8451,22556,1504.14,0.0594,0.5742
1,0.6,0.7373,0.7186,0.3319,0.4541,0.6759,3816,1494,7680,21935,1504.14,0.0594,0.5742
2,0.5,0.4613,0.3714,0.9188,0.5289,0.6759,10562,17880,934,5549,1504.14,0.0594,0.5742
3,0.4,0.4451,0.3682,0.9582,0.5320,0.6759,11016,18899,480,4530,1504.14,0.0594,0.5742
4,0.3,0.4109,0.3563,0.9796,0.5226,0.6759,11262,20342,234,3087,1504.14,0.0594,0.5742
5,0.2,0.3839,0.3475,0.9930,0.5148,0.6759,11415,21435,81,1994,1504.14,0.0594,0.5742
6,0.1,0.3508,0.3363,0.9989,0.5032,0.6759,11483,22661,13,768,1504.14,0.0594,0.5742


# Consolidación de resultados

In [ ]:


#Nombrar cada modelo
tabla_resultados_xgb["Modelo"] = "XGBoost"
tabla_resultados_gb["Modelo"] = "GradientBoosting"
tabla_resultados_rf["Modelo"] = "RandomForest"
tabla_resultados_dt["Modelo"] = "DecisionTree"
tabla_resultados_lr["Modelo"] = "LogisticRegression"



tabla_final_modelos = pd.concat([
    tabla_resultados_xgb,
    tabla_resultados_gb,
    tabla_resultados_rf,
    tabla_resultados_dt,
    tabla_resultados_lr
], ignore_index=True)


#Añadir todo en una sola tabla
columnas = ["Modelo", "Umbral", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC","TP","FP","FN","TN","Training Time (s)","Inference Time (s)","Mejor F1 promedio (CV Realista)"]
tabla_final_modelos = tabla_final_modelos[columnas]
tabla_final_modelos = tabla_final_modelos.round(4)



In [ ]:
tabla_final_modelos

,Modelo,Umbral,Accuracy,Precision,Recall,F1,ROC-AUC,TP,FP,FN,TN,Training Time (s),Inference Time (s),Mejor F1 promedio (CV Realista)
0,XGBoost,0.7,0.7330,0.7772,0.2649,0.3951,0.6759,3045,873,8451,22556,1504.14,0.0594,0.5742
1,XGBoost,0.6,0.7373,0.7186,0.3319,0.4541,0.6759,3816,1494,7680,21935,1504.14,0.0594,0.5742
2,XGBoost,0.5,0.4613,0.3714,0.9188,0.5289,0.6759,10562,17880,934,5549,1504.14,0.0594,0.5742
3,XGBoost,0.4,0.4451,0.3682,0.9582,0.5320,0.6759,11016,18899,480,4530,1504.14,0.0594,0.5742
4,XGBoost,0.3,0.4109,0.3563,0.9796,0.5226,0.6759,11262,20342,234,3087,1504.14,0.0594,0.5742
5,XGBoost,0.2,0.3839,0.3475,0.9930,0.5148,0.6759,11415,21435,81,1994,1504.14,0.0594,0.5742
6,XGBoost,0.1,0.3508,0.3363,0.9989,0.5032,0.6759,11483,22661,13,768,1504.14,0.0594,0.5742
7,GradientBoosting,0.7,0.7390,0.7518,0.3091,0.4380,0.6711,3553,1173,7943,22256,2752.09,0.0399,0.5697
8,GradientBoosting,0.6,0.7394,0.7292,0.3316,0.4559,0.6711,3812,1416,7684,22013,2752.09,0.0399,0.5697
9,GradientBoosting,0.5,0.4665,0.3732,0.9140,0.5300,0.6711,10507,17645,989,5784,2752.09,0.0399,0.5697


In [ ]:
# Ruta del destino (ajústala exactamente a cómo se llama tu carpeta)
ruta_salida = '/content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Datasets/tabla_final_modelos.csv'

# Guardar el archivo CSV
tabla_final_modelos.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

print(f"✅ Archivo guardado correctamente en:\n{ruta_salida}")

✅ Archivo guardado correctamente en:
/content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Datasets/tabla_final_modelos.csv


In [ ]:
# ============================================
# 1) DEFINICIÓN DE LA FUNCIÓN DE EXTRACCIÓN
# ============================================

def extraer_importancia_con_nomenclatura(modelo, modelo_nombre, feature_names):
    """
    Extrae coeficientes para modelos lineales (RL) e importancia para otros modelos,
    manteniendo la nomenclatura definida por el usuario.
    """

    # Acceder al estimador final dentro del Pipeline
    final_estimator = modelo.named_steps['model']

    if modelo_nombre == 'LogisticRegression':
        # Para RL: extraemos coeficientes

        importancias = final_estimator.coef_[0] if final_estimator.coef_.ndim > 1 else final_estimator.coef_
        metrica = 'Coeficiente'
    else:
        # Para Árboles (RF, GB, XGB): extraemos feature_importances_
        importancias = final_estimator.feature_importances_
        metrica = 'Importancia'

    # Crear un DataFrame con los resultados
    df_importancia = pd.DataFrame({
        'Variable': feature_names,
        'Valor_Impacto': importancias,
        'Modelo': modelo_nombre,
        'Tipo_Metrica': metrica
    })

    # Calcular el valor absoluto para ranking (esencial para RL)
    df_importancia['Valor_Absoluto'] = df_importancia['Valor_Impacto'].abs()

    # Ordenar por el valor absoluto del impacto
    df_importancia = df_importancia.sort_values(by='Valor_Absoluto', ascending=False)

    return df_importancia[['Modelo', 'Tipo_Metrica', 'Variable', 'Valor_Impacto', 'Valor_Absoluto']]

# ============================================
# 2) EJECUCIÓN Y CONSOLIDACIÓN
# ============================================

# 1. Obtener nombres de variables
try:
    feature_names = X_train.columns.tolist()
except AttributeError:
    print("⚠️ Advertencia: X_train no es un DataFrame. Asegúrate de obtener los nombres de las variables correctamente.")
    # Si X_train es un array, debes reemplazar esto con los nombres reales de tus variables
    feature_names = [f'feature_{i}' for i in range(X_train.shape[1])]


# 2. Extracción de importancia/coeficientes
# NOTA: Debes ejecutar esto después de entrenar CADA modelo (LR, RF, GB, XGB) y tener definidos best_lr, best_rf, etc.
# Ejemplo de uso:
df_lr = extraer_importancia_con_nomenclatura(best_lr, 'LogisticRegression', feature_names)
df_dt = extraer_importancia_con_nomenclatura(best_dt, 'DecisionTree', feature_names)
df_rf = extraer_importancia_con_nomenclatura(best_rf, 'RandomForest', feature_names)
df_gb = extraer_importancia_con_nomenclatura(best_gb, 'GradientBoosting', feature_names)
df_xgb = extraer_importancia_con_nomenclatura(best_xgb, 'XGBoost', feature_names)


# 3. Consolidar todos los resultados en un único DataFrame
df_consolidados = [df_lr, df_dt, df_rf, df_gb, df_xgb]

df_interpretabilidad_final = pd.concat(df_consolidados, ignore_index=True)


# Ruta del destino (ajústala exactamente a cómo se llama tu carpeta)
ruta_salida_fi = '/content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Datasets/tabla_final_importancia_variables.csv'

# Guardar el archivo CSV
df_interpretabilidad_final.to_csv(ruta_salida_fi, index=False, encoding='utf-8-sig')

print(f"✅ Archivo guardado correctamente en:\n{ruta_salida}")

✅ Archivo guardado correctamente en:
/content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Datasets/tabla_final_modelos.csv


In [ ]:
df_interpretabilidad_final

,Modelo,Tipo_Metrica,Variable,Valor_Impacto,Valor_Absoluto
0,LogisticRegression,Coeficiente,artritis,-2.536510,2.536510
1,LogisticRegression,Coeficiente,asma,-2.385305,2.385305
2,LogisticRegression,Coeficiente,huerfanas_hemofilias_y_otras,-2.150506,2.150506
3,LogisticRegression,Coeficiente,ciclo_de_vida_PERSONA MAYOR,1.970266,1.970266
4,LogisticRegression,Coeficiente,trasplantados,-1.756721,1.756721
...,...,...,...,...,...
145,XGBoost,Importancia,tipo_erc_limpio_Estadio V,0.000000,0.000000
146,XGBoost,Importancia,erc_trr_limpio_Recibe TRR (no especificada),0.000000,0.000000
147,XGBoost,Importancia,erc_trr_limpio_Prediálisis,0.000000,0.000000
148,XGBoost,Importancia,regimen_E,0.000000,0.000000


# Guardamos los modelos

In [ ]:
import joblib
import os
import time

# --- ⚠️ RUTA DEFINIDA POR EL USUARIO ⚠️ ---
# La ruta en Drive debe existir
BASE_PATH = '/content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Modelos Guardados'

# 1. Asegurar que el directorio exista (crearlo si es necesario)
os.makedirs(BASE_PATH, exist_ok=True)
print(f"Directorio de guardado verificado: {BASE_PATH}")

# 2. Diccionario de modelos y sus nombres de archivo
# ASUMIMOS QUE ESTOS OBJETOS YA ESTÁN CARGADOS/ENTRENADOS EN MEMORIA
modelos_a_guardar = {
    'LogisticRegression': best_lr,
    'DecisionTree': best_dt,
    'RandomForest': best_rf,
    'GradientBoosting': best_gb,
    'XGBoost': best_xgb
}

# 3. Iterar y guardar cada modelo
for nombre, modelo in modelos_a_guardar.items():

    # 3.1 Construir el nombre completo del archivo
    filename = os.path.join(BASE_PATH, f'{nombre}_pipeline_final.joblib')

    # 3.2 Guardar el modelo
    start_save = time.time()
    joblib.dump(modelo, filename)
    end_save = time.time()

    print(f"✅ Modelo '{nombre}' guardado en {filename}")
    print(f"   Tiempo de guardado: {end_save - start_save:.4f} segundos.")

print("\n🎉 ¡Todos los modelos han sido guardados exitosamente en Google Drive!")

Directorio de guardado verificado: /content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Modelos Guardados
✅ Modelo 'LogisticRegression' guardado en /content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Modelos Guardados/LogisticRegression_pipeline_final.joblib
   Tiempo de guardado: 0.5520 segundos.
✅ Modelo 'DecisionTree' guardado en /content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Modelos Guardados/DecisionTree_pipeline_final.joblib
   Tiempo de guardado: 0.2629 segundos.
✅ Modelo 'RandomForest' guardado en /content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Modelos Guardados/RandomForest_pipeline_final.joblib
   Tiempo de guardado: 0.4258 segundos.
✅ Modelo 'GradientBoosting' guardado en /content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Modelos Guard

#Para interpretar resultados de RL

In [ ]:


# ============================================
# 1) EXTRACCIÓN Y CÁLCULO DE ODS RATIOS
# ============================================

# Acceder al estimador de Regresión Logística dentro del Pipeline
lr_estimator = best_lr.named_steps['model']

# 1. Extraer los Coeficientes (Beta)
# Manejamos si coef_ es 1D o 2D (toma el primer array para la clase 1)
coeficientes = lr_estimator.coef_[0] if lr_estimator.coef_.ndim > 1 else lr_estimator.coef_

# 2. Calcular los Odds Ratios (e^Beta)
odds_ratios = np.exp(coeficientes)

# 3. Crear el DataFrame
df_odds_ratios = pd.DataFrame({
    'Variable': feature_names,
    'Coeficiente (Beta)': coeficientes,
    'Odds Ratio (OR)': odds_ratios
})

# 4. Ordenar por Odds Ratio (los mayores riesgos arriba)
df_odds_ratios = df_odds_ratios.sort_values(by='Odds Ratio (OR)', ascending=False).reset_index(drop=True)


# 5. Exportar a CSV (Ideal para tu Looker/Visualización)
nombre_archivo_ods = 'odds_ratios_logistic_regression.csv'


# Ruta del destino (ajústala exactamente a cómo se llama tu carpeta)
ruta_salida_ods_rl = '/content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Datasets/tabla_ods_rl.csv'

# Guardar el archivo CSV
df_odds_ratios.to_csv(ruta_salida_ods_rl, index=False, encoding='utf-8-sig')

print(f"✅ Archivo guardado correctamente en:\n{ruta_salida}")

✅ Archivo guardado correctamente en:
/content/drive/MyDrive/1. Proyectos/e. Proyecto Predicción de enfermedades crónicas en Bucaramanga/Datasets/tabla_final_modelos.csv


In [ ]:
df_odds_ratios

,Variable,Coeficiente (Beta),Odds Ratio (OR)
0,ciclo_de_vida_PERSONA MAYOR,1.970266,7.172587
1,ciclo_de_vida_ADULTEZ,1.374471,3.952986
2,ciclo_de_vida_PRIMERA INFANCIA,1.195125,3.303969
3,regimen_P,0.852376,2.345214
4,compromiso_renal,0.794581,2.213514
5,tipo_erc_limpio_SIN CLAS,0.581011,1.787846
6,tipo_erc_limpio_Estadio IV,0.269728,1.309609
7,regimen_S,0.206697,1.229610
8,ciclo_de_vida_JOVENES,0.172219,1.187938
9,insuficiencia_cardiaca,0.124349,1.132411
